<a href="https://colab.research.google.com/github/dilaracb/yz-odev/blob/main/yzproje.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

!pip install wordcloud -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from wordcloud import WordCloud
import warnings
warnings.filterwarnings('ignore')

# Grafik stili
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11
sns.set_theme(style='whitegrid')

print('Kütüphaneler hazıd')

In [ ]:

df = pd.read_csv('fake reviews dataset.csv', engine='python', on_bad_lines='skip')

print('Dataset yüklendi!')
print(f'Boyut : {df.shape[0]:,} satır × {df.shape[1]} sütun')
print(f'\n Sütunlar: {list(df.columns)}')

In [ ]:
print('── İLK 5 SATIR ───')
display(df.head())

print('\n── VERİ TİPLERİ ───')
print(df.dtypes)

print('\n── EKSİK DEĞERLER ───')
missing = df.isnull().sum()
print(missing)
if missing.sum() == 0:
    print('Eksik değer yok!')
else:
    print(f'Toplam {missing.sum()} eksik değer var.')

print('\n── TEKRAR EDEN SATIRLAR ───')
dup = df.duplicated().sum()
print(f'Tekrarlı satır sayısı: {dup}')
if dup > 0:
    df = df.drop_duplicates()
    print(f'Tekrarlar temizlendi. Yeni boyut: {df.shape}')

In [ ]:
# Etiket sütunu adını bulalım
label_col = 'label' if 'label' in df.columns else df.columns[-1]
text_col  = 'text_' if 'text_' in df.columns else 'text'
print(f'Etiket sütunu : {label_col}')
print(f'Metin sütunu  : {text_col}')

counts = df[label_col].value_counts()
print(f'\n── ETİKET DAĞILIMI ───')
for label, count in counts.items():
    bar = '█' * int(count / 200)
    print(f'{label:>5}  {bar}  {count:,} yorum  (%{count/len(df)*100:.1f})')

# Grafik
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar grafik
colors = ['#ef4444', '#22c55e']
axes[0].bar(counts.index, counts.values, color=colors, width=0.5, edgecolor='white', linewidth=1.5)
axes[0].set_title('Yorum Dağılımı (Sayı)', fontweight='bold')
axes[0].set_ylabel('Yorum Sayısı')
for i, (label, val) in enumerate(counts.items()):
    axes[0].text(i, val + 100, f'{val:,}', ha='center', fontweight='bold')

# Pasta grafik
axes[1].pie(
    counts.values,
    labels=['CG (Sahte)', 'OR (Gerçek)'],
    colors=colors,
    autopct='%1.1f%%',
    startangle=90,
    wedgeprops=dict(edgecolor='white', linewidth=2)
)
axes[1].set_title('Yorum Dağılımı (Oran)', fontweight='bold')

plt.suptitle('Etiket Dağılımı: Sahte vs Gerçek', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('etiket_dagilimi.png', bbox_inches='tight')
plt.show()


In [ ]:
# Uzunluk hesaplayalım
df['yorum_uzunlugu']   = df[text_col].astype(str).apply(len)
df['kelime_sayisi']    = df[text_col].astype(str).apply(lambda x: len(x.split()))

print('── UZUNLUK İSTATİSTİKLERİ ───')
stats = df.groupby(label_col)[['yorum_uzunlugu', 'kelime_sayisi']].describe().round(1)
display(stats)

# Grafik
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
label_values = df[label_col].unique()
colors_map   = {label_values[0]: '#ef4444', label_values[1]: '#22c55e'}

for lbl in label_values:
    subset = df[df[label_col] == lbl]
    color  = colors_map[lbl]
    label_name = 'CG (Sahte)' if lbl == 'CG' else 'OR (Gerçek)'

    axes[0].hist(subset['yorum_uzunlugu'].clip(upper=2000),
                 bins=50, alpha=0.6, color=color, label=label_name, edgecolor='none')
    axes[1].hist(subset['kelime_sayisi'].clip(upper=400),
                 bins=50, alpha=0.6, color=color, label=label_name, edgecolor='none')

axes[0].set_title('Karakter Uzunluğu Dağılımı', fontweight='bold')
axes[0].set_xlabel('Karakter Sayısı')
axes[0].set_ylabel('Yorum Sayısı')
axes[0].legend()

axes[1].set_title('Kelime Sayısı Dağılımı', fontweight='bold')
axes[1].set_xlabel('Kelime Sayısı')
axes[1].set_ylabel('Yorum Sayısı')
axes[1].legend()

plt.suptitle('Yorum Uzunluğu: Sahte vs Gerçek', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('uzunluk_analizi.png', bbox_inches='tight')
plt.show()


In [ ]:
if 'rating' in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for i, lbl in enumerate(label_values):
        subset = df[df[label_col] == lbl]
        color  = colors_map[lbl]
        label_name = 'CG (Sahte)' if lbl == 'CG' else 'OR (Gerçek)'
        rating_counts = subset['rating'].value_counts().sort_index()

        axes[i].bar(rating_counts.index, rating_counts.values,
                    color=color, edgecolor='white', linewidth=1.5)
        axes[i].set_title(f'Puan Dağılımı — {label_name}', fontweight='bold')
        axes[i].set_xlabel('Puan (1-5 ⭐)')
        axes[i].set_ylabel('Yorum Sayısı')
        axes[i].set_xticks([1, 2, 3, 4, 5])
        for x, y in zip(rating_counts.index, rating_counts.values):
            axes[i].text(x, y + 50, str(y), ha='center', fontsize=9)

    plt.suptitle('Puan Dağılımı: Sahte vs Gerçek', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('puan_dagilimi.png', bbox_inches='tight')
    plt.show()


    print('\n── PUAN ORTALAMASI ──')
    print(df.groupby(label_col)['rating'].mean().round(2))
else:
    print('rating sütunu bu datasette yok, hücre atlandı.') #önlem

In [ ]:
if 'category' in df.columns:
    print('── KATEGORİLER ───')
    print(df['category'].value_counts())

    fig, ax = plt.subplots(figsize=(12, 5))
    cat_counts = df['category'].value_counts()
    bars = ax.barh(cat_counts.index[:15], cat_counts.values[:15],
                   color='#6366f1', edgecolor='white')
    ax.set_title('En Çok Yorum Alan Kategoriler (İlk 15)', fontweight='bold')
    ax.set_xlabel('Yorum Sayısı')
    for bar, val in zip(bars, cat_counts.values[:15]):
        ax.text(val + 30, bar.get_y() + bar.get_height()/2,
                str(val), va='center', fontsize=9)
    plt.tight_layout()
    plt.savefig('kategori_dagilimi.png', bbox_inches='tight')
    plt.show()
    print('Grafik kaydedildi, kategori_dagilimi.png')
else:
    print('category sütunu bu datasette yok, hücre atlandı.')

In [ ]:
fake_label = 'CG' if 'CG' in df[label_col].values else df[label_col].unique()[0]
real_label = 'OR' if 'OR' in df[label_col].values else df[label_col].unique()[1]

fake_text = ' '.join(df[df[label_col] == fake_label][text_col].astype(str).tolist())
real_text = ' '.join(df[df[label_col] == real_label][text_col].astype(str).tolist())

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

wc_fake = WordCloud(
    width=800, height=400,
    background_color='white',
    colormap='Reds',
    max_words=80
).generate(fake_text)

wc_real = WordCloud(
    width=800, height=400,
    background_color='white',
    colormap='Greens',
    max_words=80
).generate(real_text)

axes[0].imshow(wc_fake, interpolation='bilinear')
axes[0].axis('off')
axes[0].set_title('🔴 CG — Sahte Yorumlar', fontsize=14, fontweight='bold', pad=12)

axes[1].imshow(wc_real, interpolation='bilinear')
axes[1].axis('off')
axes[1].set_title('🟢 OR — Gerçek Yorumlar', fontsize=14, fontweight='bold', pad=12)

plt.suptitle('Kelime Bulutu: Sahte vs Gerçek', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('kelime_bulutu.png', bbox_inches='tight')
plt.show()


In [ ]:
print('=' * 50)
print('         EDA ÖZET RAPORU')
print('=' * 50)
print(f'Toplam yorum sayısı   : {len(df):,}')
print(f'Sahte yorum (CG)      : {(df[label_col]==fake_label).sum():,}')
print(f'Gerçek yorum (OR)     : {(df[label_col]==real_label).sum():,}')
print(f'Eksik değer           : {df.isnull().sum().sum()}')
print(f'Ortalama kelime sayısı: {df["kelime_sayisi"].mean():.1f}')
print(f'  → Sahte yorumlarda  : {df[df[label_col]==fake_label]["kelime_sayisi"].mean():.1f}')
print(f'  → Gerçek yorumlarda : {df[df[label_col]==real_label]["kelime_sayisi"].mean():.1f}')
if 'category' in df.columns:
    print(f'Kategori sayısı       : {df["category"].nunique()}')
if 'rating' in df.columns:
    print(f'Ortalama puan (Sahte) : {df[df[label_col]==fake_label]["rating"].mean():.2f}')
    print(f'Ortalama puan (Gerçek): {df[df[label_col]==real_label]["rating"].mean():.2f}')
print('=' * 50)
# Temizlenmiş veriyi kaydedelim
df.to_csv('temiz_veri.csv', index=False)
print('\nTemizlenmiş veri hzır, temiz_veri.csv')

In [ ]:
!pip install transformers torch scikit-learn -q

import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from transformers import BertTokenizer
import torch
from torch.utils.data import Dataset, DataLoader
import warnings
warnings.filterwarnings('ignore')

print('Kütüphaneler hazır')
print(f'PyTorch sürümü : {torch.__version__}')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Kullanılacak cihaz: {device.upper()}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('GPU bulunamadı. Colab\'da Runtime → Change runtime type → T4 GPU seç!')

In [ ]:
df = pd.read_csv('temiz_veri.csv')

# Sütun adlarını otomatik algıla
text_col  = 'text_'  if 'text_'  in df.columns else 'text'
label_col = 'label'  if 'label'  in df.columns else df.columns[-1]

print(f'Veri yüklendi: {df.shape[0]:,} satır')
print(f'Metin sütunu  : {text_col}')
print(f'Etiket sütunu : {label_col}')
print(f'\nEtiketler: {df[label_col].unique()}')

In [ ]:
def metni_temizle(text):
    """Ham yorumu modele hazır hale getirir."""
    text = str(text)
    # HTML etiketleri kaldır
    text = re.sub(r'<.*?>', '', text)
    # URL kaldır
    text = re.sub(r'http\S+|www\S+', '', text)
    # Çok fazla tekrarlayan noktalamayı sadeleştir
    text = re.sub(r'[!]{2,}', '!', text)
    text = re.sub(r'[.]{2,}', '.', text)
    # Rakamları koru ama özel karakterleri kaldır
    text = re.sub(r'[^a-zA-Z0-9\s.,!?\'"-]', '', text)
    # Fazla boşlukları temizle
    text = re.sub(r'\s+', ' ', text).strip()
    # Küçük harfe çevir
    text = text.lower()
    return text

# Temizlemeyi uygula
df['temiz_metin'] = df[text_col].apply(metni_temizle)

# Önce / sonra karşılaştırma
print('── ÖNCE / SONRA KARŞILAŞTIRMA ─────────────────────')
for i in [0, 1, 2]:
    print(f'\n[{i+1}] Orijinal : {df[text_col].iloc[i][:100]}')
    print(f'    Temiz    : {df["temiz_metin"].iloc[i][:100]}')

# Boş kalan yorumları kaldır
bos_once = len(df)
df = df[df['temiz_metin'].str.len() > 10].reset_index(drop=True)
print(f'\n Temizleme tamamlandı!')
print(f'   Kaldırılan çok kısa yorum: {bos_once - len(df)}')
print(f'   Kalan yorum sayısı       : {len(df):,}')

In [ ]:
# CG → 1 (Sahte), OR → 0 (Gerçek)
le = LabelEncoder()
df['label_encoded'] = le.fit_transform(df[label_col])

print('  ETİKET DÖNÜŞÜMÜ ──')
for original, encoded in zip(le.classes_, le.transform(le.classes_)):
    anlam = 'SAHTE' if original == 'CG' else 'GERÇEK'
    print(f'  {original} → {encoded}  ({anlam})')

print(f'\nDağılım:')
print(df['label_encoded'].value_counts())

In [ ]:
# %80 train | %10 validation | %10 test bölme oranları
X = df['temiz_metin'].values
y = df['label_encoded'].values

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print('── VERİ BÖLME SONUCU ───')
print(f'Train seti      : {len(X_train):>6,} yorum  (%80)')
print(f'Validation seti : {len(X_val):>6,} yorum  (%10)')
print(f'Test seti       : {len(X_test):>6,} yorum  (%10)')

print('\n── SINIF DENGESİ KONTROLÜ ──')
for name, labels in [('Train', y_train), ('Val', y_val), ('Test', y_test)]:
    sahte  = labels.sum()
    gercek = len(labels) - sahte
    print(f'{name:>5}: Sahte={sahte:,}  Gerçek={gercek:,}')

# Görsel
fig, ax = plt.subplots(figsize=(8, 4))
sets   = ['Train', 'Validation', 'Test']
sizes  = [len(X_train), len(X_val), len(X_test)]
colors = ['#4361ee', '#7209b7', '#f72585']
bars   = ax.bar(sets, sizes, color=colors, width=0.5, edgecolor='white')
for bar, size in zip(bars, sizes):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
            f'{size:,}', ha='center', fontweight='bold')
ax.set_title('Veri Seti Bölme Dağılımı', fontweight='bold')
ax.set_ylabel('Yorum Sayısı')
plt.tight_layout()
plt.savefig('veri_bolme.png', bbox_inches='tight')
plt.show()


In [ ]:
print('BERT tokenizer yükle')
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
print('Tokenizer hazır')

# Optimum MAX_LEN bul
sample_lengths = [len(tokenizer.encode(text)) for text in X_train[:2000]]
print(f'\n── TOKEN UZUNLUĞU ANALİZİ (ilk 2000 yorum) ──')
print(f'Ortalama  : {np.mean(sample_lengths):.0f} token')
print(f'Medyan    : {np.median(sample_lengths):.0f} token')
print(f'%95 perct : {np.percentile(sample_lengths, 95):.0f} token')
print(f'Maksimum  : {max(sample_lengths)} token')

# Histogram
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(sample_lengths, bins=50, color='#4361ee', edgecolor='none', alpha=0.8)
ax.axvline(128, color='#ef4444', linestyle='--', linewidth=2, label='MAX_LEN = 128')
ax.axvline(np.percentile(sample_lengths, 95), color='#f59e0b',
           linestyle='--', linewidth=2, label='%95 persentil')
ax.set_title('Token Uzunluğu Dağılımı', fontweight='bold')
ax.set_xlabel('Token Sayısı')
ax.set_ylabel('Yorum Sayısı')
ax.legend()
plt.tight_layout()
plt.savefig('token_uzunlugu.png', bbox_inches='tight')
plt.show()


In [ ]:
MAX_LEN    = 128   # Token analizi sonrası gerekirse 256 yap
BATCH_SIZE = 16

class YorumDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length      = self.max_len,
            padding         = 'max_length',
            truncation      = True,
            return_tensors  = 'pt'
        )
        return {
            'input_ids'      : encoding['input_ids'].squeeze(),
            'attention_mask' : encoding['attention_mask'].squeeze(),
            'label'          : torch.tensor(self.labels[idx], dtype=torch.long)
        }

# DataLoader'ları oluştur
train_dataset = YorumDataset(X_train, y_train, tokenizer, MAX_LEN)
val_dataset   = YorumDataset(X_val,   y_val,   tokenizer, MAX_LEN)
test_dataset  = YorumDataset(X_test,  y_test,  tokenizer, MAX_LEN)

train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader    = DataLoader(val_dataset,   batch_size=BATCH_SIZE)
test_loader   = DataLoader(test_dataset,  batch_size=BATCH_SIZE)

print('── DATALOADER ÖZET ──────────────────────')
print(f'MAX_LEN        : {MAX_LEN} token')
print(f'Batch size     : {BATCH_SIZE}')
print(f'Train batch    : {len(train_loader):,}')
print(f'Val batch      : {len(val_loader):,}')
print(f'Test batch     : {len(test_loader):,}')

# Test et
sample_batch = next(iter(train_loader))
print(f'\nBatch şekli    : {sample_batch["input_ids"].shape}')
print('DataLoader hazır')

In [ ]:
import pickle

# CSV olarak kaydet (yedek)
pd.DataFrame({'text': X_train, 'label': y_train}).to_csv('train.csv', index=False)
pd.DataFrame({'text': X_val,   'label': y_val  }).to_csv('val.csv',   index=False)
pd.DataFrame({'text': X_test,  'label': y_test }).to_csv('test.csv',  index=False)

# DataLoader'ları kaydet
with open('loaders.pkl', 'wb') as f:
    pickle.dump({
        'train_loader' : train_loader,
        'val_loader'   : val_loader,
        'test_loader'  : test_loader,
        'label_encoder': le
    }, f)

print('═' * 50)
print('      AŞAMA 2 ÖZET RAPORU')
print('═' * 50)
print(f'Temizleme      : Tamamlandı')
print(f'Encoding       : CG=1 (Sahte) | OR=0 (Gerçek)')
print(f'Train seti     : {len(X_train):,} yorum')
print(f'Val seti       : {len(X_val):,} yorum')
print(f'Test seti      : {len(X_test):,} yorum')
print(f'MAX_LEN        : {MAX_LEN} token')
print(f'Batch size     : {BATCH_SIZE}')
print(f'Cihaz          : {device.upper()}')
print('═' * 50)


In [ ]:
!pip install transformers torch scikit-learn -q

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertModel, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import time, warnings
warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 120

# GPU kontrolü
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Kullanılan cihaz : {device}')
if device.type == 'cuda':
    print(f'GPU modeli       : {torch.cuda.get_device_name(0)}')
    print(f'GPU belleği      : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('GPU YOK! Runtime → Change runtime type → T4 GPU seç!')

# Tekrarlanabilirlik için seed
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
print('\nKütüphaneler hazır')

In [ ]:
# ── TÜM AYARLAR BURADA ────────────────────────────────────
MODEL_NAME  = 'bert-base-uncased'  # Kullanılacak BERT modeli
MAX_LEN     = 128                  # Maksimum token uzunluğu
BATCH_SIZE  = 16                   # Her seferinde kaç yorum
EPOCHS      = 4                    # Kaç tur eğitim
LR          = 2e-5                 # Öğrenme hızı
DROPOUT     = 0.3                  # Dropout oranı
NUM_CLASSES = 2                    # Sahte / Gerçek
# ──────────────────────────────────────────────────────────

print('── HİPERPARAMETRELER ──')
print(f'Model        : {MODEL_NAME}')
print(f'MAX_LEN      : {MAX_LEN}')
print(f'Batch size   : {BATCH_SIZE}')
print(f'Epoch sayısı : {EPOCHS}')
print(f'Learning rate: {LR}')
print(f'Dropout      : {DROPOUT}')

In [ ]:
train_df = pd.read_csv('train.csv')
val_df   = pd.read_csv('val.csv')
test_df  = pd.read_csv('test.csv')

X_train, y_train = train_df['text'].values, train_df['label'].values
X_val,   y_val   = val_df['text'].values,   val_df['label'].values
X_test,  y_test  = test_df['text'].values,  test_df['label'].values

print('Veriler hazıe')
print(f'Train : {len(X_train):,} yorum')
print(f'Val   : {len(X_val):,} yorum')
print(f'Test  : {len(X_test):,} yorum')

In [ ]:
print('Tokenizer yükleniyor...')
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)
print('Tokenizer hazır')

class YorumDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            str(self.texts[idx]),
            max_length     = self.max_len,
            padding        = 'max_length',
            truncation     = True,
            return_tensors = 'pt'
        )
        return {
            'input_ids'     : enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
            'label'         : torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_loader = DataLoader(
    YorumDataset(X_train, y_train, tokenizer, MAX_LEN),
    batch_size=BATCH_SIZE, shuffle=True
)
val_loader = DataLoader(
    YorumDataset(X_val, y_val, tokenizer, MAX_LEN),
    batch_size=BATCH_SIZE
)
test_loader = DataLoader(
    YorumDataset(X_test, y_test, tokenizer, MAX_LEN),
    batch_size=BATCH_SIZE
)

print(f'Train batch sayısı : {len(train_loader)}')
print(f'Val batch sayısı   : {len(val_loader)}')
print('DataLoader hazır')

In [ ]:
class SahteYorumDetector(nn.Module):
    """
    Fine-tuned BERT for Binary Classification

    Yapı:
    BERT Encoder (768 boyut)
        --> Dropout
        --> Linear(768 → 256)
        --> ReLU
        --> Dropout
        --> Linear(256 → 2)
        --> Softmax → [Gerçek olasılığı, Sahte olasılığı]
    """
    def __init__(self, model_name, num_classes, dropout):
        super(SahteYorumDetector, self).__init__()
        self.bert    = BertModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(dropout)
        self.fc1     = nn.Linear(768, 256)
        self.relu    = nn.ReLU()
        self.fc2     = nn.Linear(256, num_classes)

    def forward(self, input_ids, attention_mask):
        # BERT çıktısı: (batch, seq_len, 768)
        output       = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        # [CLS] tokeni → cümle temsili (batch, 768)
        cls_output   = output.pooler_output
        x            = self.dropout(cls_output)
        x            = self.fc1(x)
        x            = self.relu(x)
        x            = self.dropout(x)
        logits       = self.fc2(x)
        return logits

print('Model yükleniyor (1-2 dakika sürebilir)...')
model = SahteYorumDetector(MODEL_NAME, NUM_CLASSES, DROPOUT).to(device)

# Parametre sayısı
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nModel hazır')
print(f'Toplam parametre    : {total_params:,}')
print(f'Eğitilebilir param. : {trainable_params:,}')

In [ ]:
# Loss fonksiyonu
criterion = nn.CrossEntropyLoss()

# Optimizer — BERT için AdamW önerilir
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)

# Scheduler — öğrenme hızını epoch boyunca kademeli düşürür
total_steps    = len(train_loader) * EPOCHS
warmup_steps   = total_steps // 10   # İlk %10'da ısınma
scheduler      = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps   = warmup_steps,
    num_training_steps = total_steps
)

print('── OPTİMİZER AYARLARI ───────────────────────')
print(f'Optimizer     : AdamW')
print(f'Loss          : CrossEntropyLoss')
print(f'Scheduler     : Linear warmup + decay')
print(f'Toplam adım   : {total_steps:,}')
print(f'Warmup adım   : {warmup_steps:,}')

In [ ]:
def egit_bir_epoch(model, loader, optimizer, scheduler, criterion, device):
    model.train()
    toplam_loss, dogru, toplam = 0, 0, 0
    for batch in loader:
        input_ids  = batch['input_ids'].to(device)
        attn_mask  = batch['attention_mask'].to(device)
        labels     = batch['label'].to(device)

        optimizer.zero_grad()
        logits     = model(input_ids, attn_mask)
        loss       = criterion(logits, labels)
        loss.backward()

        # Gradient clipping — patlamayı önler
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        scheduler.step()

        toplam_loss += loss.item()
        tahminler    = logits.argmax(dim=1)
        dogru       += (tahminler == labels).sum().item()
        toplam      += len(labels)

    return toplam_loss / len(loader), dogru / toplam


def degerlendir(model, loader, criterion, device):
    model.eval()
    toplam_loss, dogru, toplam = 0, 0, 0
    tum_tahmin, tum_gercek     = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            attn_mask = batch['attention_mask'].to(device)
            labels    = batch['label'].to(device)

            logits    = model(input_ids, attn_mask)
            loss      = criterion(logits, labels)
            tahminler = logits.argmax(dim=1)

            toplam_loss += loss.item()
            dogru       += (tahminler == labels).sum().item()
            toplam      += len(labels)
            tum_tahmin.extend(tahminler.cpu().numpy())
            tum_gercek.extend(labels.cpu().numpy())

    return toplam_loss / len(loader), dogru / toplam, tum_tahmin, tum_gercek


# ── EĞİTİM BAŞLIYOR ───────────────────────────────────────────────────
gecmis = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
en_iyi_val_acc  = 0
en_iyi_model_yolu = 'en_iyi_model.pt'

print('🚀 EĞİTİM BAŞLADI')
print('=' * 60)

for epoch in range(1, EPOCHS + 1):
    baslangic = time.time()

    train_loss, train_acc = egit_bir_epoch(
        model, train_loader, optimizer, scheduler, criterion, device
    )
    val_loss, val_acc, _, _ = degerlendir(model, val_loader, criterion, device)

    sure = time.time() - baslangic

    gecmis['train_loss'].append(train_loss)
    gecmis['val_loss'].append(val_loss)
    gecmis['train_acc'].append(train_acc)
    gecmis['val_acc'].append(val_acc)

    # En iyi modeli kaydet
    if val_acc > en_iyi_val_acc:
        en_iyi_val_acc = val_acc
        torch.save(model.state_dict(), en_iyi_model_yolu)
        kayit_notu = '  ✅ Model kaydedildi!'
    else:
        kayit_notu = ''

    print(f'Epoch {epoch}/{EPOCHS} | '
          f'Train Loss: {train_loss:.4f} | Train Acc: %{train_acc*100:.2f} | '
          f'Val Loss: {val_loss:.4f} | Val Acc: %{val_acc*100:.2f} | '
          f'Süre: {sure:.0f}s{kayit_notu}')

print('=' * 60)
print(f'🏆 En iyi Val Accuracy: %{en_iyi_val_acc*100:.2f}')

In [ ]:
epochs_x = range(1, EPOCHS + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss grafiği
axes[0].plot(epochs_x, gecmis['train_loss'], 'o-', color='#4361ee',
             linewidth=2, markersize=7, label='Train Loss')
axes[0].plot(epochs_x, gecmis['val_loss'],   's--', color='#ef4444',
             linewidth=2, markersize=7, label='Val Loss')
axes[0].set_title('Epoch Başına Loss', fontweight='bold', fontsize=13)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].set_xticks(epochs_x)
axes[0].grid(True, alpha=0.3)

# Accuracy grafiği
axes[1].plot(epochs_x, [a*100 for a in gecmis['train_acc']], 'o-',
             color='#4361ee', linewidth=2, markersize=7, label='Train Acc')
axes[1].plot(epochs_x, [a*100 for a in gecmis['val_acc']], 's--',
             color='#22c55e', linewidth=2, markersize=7, label='Val Acc')
axes[1].set_title('Epoch Başına Accuracy', fontweight='bold', fontsize=13)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].legend()
axes[1].set_xticks(epochs_x)
axes[1].grid(True, alpha=0.3)

plt.suptitle('Eğitim Süreci', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('egitim_grafik.png', bbox_inches='tight')
plt.show()

In [ ]:
# En iyi modeli yükle
model.load_state_dict(torch.load(en_iyi_model_yolu))
_, test_acc, tahminler, gercekler = degerlendir(model, test_loader, criterion, device)

acc  = accuracy_score(gercekler, tahminler)
prec = precision_score(gercekler, tahminler)
rec  = recall_score(gercekler, tahminler)
f1   = f1_score(gercekler, tahminler)

print('═' * 50)
print('      TEST SETİ SONUÇLARI')
print('═' * 50)
print(f'Accuracy  : %{acc*100:.2f}')
print(f'Precision : %{prec*100:.2f}')
print(f'Recall    : %{rec*100:.2f}')
print(f'F1-Score  : %{f1*100:.2f}')
print('═' * 50)
print('\n── SINIFLANDIRMA RAPORU ───')
print(classification_report(gercekler, tahminler,
                             target_names=['Gerçek (OR)', 'Sahte (CG)']))

In [ ]:
cm = confusion_matrix(gercekler, tahminler)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=['Gerçek (Tahmin)', 'Sahte (Tahmin)'],
    yticklabels=['Gerçek (Gerçek)', 'Sahte (Gerçek)'],
    ax=axes[0], annot_kws={'size': 16, 'weight': 'bold'}
)
axes[0].set_title('Confusion Matrix', fontweight='bold', fontsize=13)
axes[0].set_ylabel('Gerçek Etiket')
axes[0].set_xlabel('Tahmin Edilen Etiket')

# Metrik karşılaştırma
metrikler = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
degerler  = [acc*100, prec*100, rec*100, f1*100]
renkler   = ['#4361ee', '#7209b7', '#f72585', '#22c55e']
bars = axes[1].bar(metrikler, degerler, color=renkler,
                   width=0.5, edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, degerler):
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.3,
                 f'%{val:.2f}', ha='center', fontweight='bold', fontsize=11)
axes[1].set_ylim(0, 105)
axes[1].set_title('Model Performans Metrikleri', fontweight='bold', fontsize=13)
axes[1].set_ylabel('Değer (%)')
axes[1].grid(True, alpha=0.3, axis='y')

plt.suptitle('Model Değerlendirme Sonuçları', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('model_sonuclar.png', bbox_inches='tight')
plt.show()

In [ ]:
import pickle

# Model ağırlıklarını kaydet
torch.save(model.state_dict(), 'bert_fake_review_model.pt')

# Tokenizer'ı kaydet
tokenizer.save_pretrained('./tokenizer')

# Metrikleri kaydet (Streamlit arayüzü için)
metrik_sonuclari = {
    'accuracy' : round(acc, 4),
    'precision': round(prec, 4),
    'recall'   : round(rec, 4),
    'f1'       : round(f1, 4),
    'egitim_gecmis': gecmis
}
with open('metrikler.pkl', 'wb') as f:
    pickle.dump(metrik_sonuclari, f)

!cp bert_fake_review_model.pt /content/drive/MyDrive/

print('═' * 50)
print('       AŞAMA 3 ÖZET RAPORU')
print('═' * 50)
print(f'Model          : {MODEL_NAME} (fine-tuned)')
print(f'Epoch          : {EPOCHS}')
print(f'Test Accuracy  : %{acc*100:.2f}')
print(f'Test F1-Score  : %{f1*100:.2f}')
print(f'Kaydedilen     : bert_fake_review_model.pt')
print('═' * 50)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Hücre C — Gerçek dataset örnekleriyle test
import pandas as pd

df_test = pd.read_csv('test.csv')

# Her sınıftan 3'er örnek al
sahte_ornekler  = df_test[df_test['label'] == 0].head(3)  # CG
gercek_ornekler = df_test[df_test['label'] == 1].head(3)  # OR
test_ornekleri  = pd.concat([sahte_ornekler, gercek_ornekler])

print('── GERÇEK DATASET ÖRNEKLERİYLE TEST ────────')
print()
for _, row in test_ornekleri.iterrows():
    gercek_etiket = '🔴 SAHTE (CG)' if row['label'] == 0 else '🟢 GERÇEK (OR)'
    sonuc = tahmin_et(row['text'], model, tokenizer, MAX_LEN, device)
    tahmin_etiket = '🔴 SAHTE' if sonuc['karar'] == 'SAHTE' else '🟢 GERÇEK'
    dogru = '✅' if gercek_etiket[:2] == tahmin_etiket[:2] else '❌'

    print(f'Metin   : {str(row["text"])[:80]}...')
    print(f'Gerçek  : {gercek_etiket}')
    print(f'Tahmin  : {tahmin_etiket} | '
          f'Sahte: %{sonuc["sahte_olasilik"]} | '
          f'Gerçek: %{sonuc["gercek_olasilik"]} {dogru}')
    print()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# Drive'da model dosyalarını ara
for root, dirs, files in os.walk('/content/drive/MyDrive'):
    for file in files:
        if file.endswith('.pt') or file.endswith('.pkl'):
            print(os.path.join(root, file))

In [ ]:
# Hücre 3 — Tokenizer'ı yeniden indir (1 dakika sürer)
from transformers import BertTokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
tokenizer.save_pretrained('./tokenizer')
print('✅ Tokenizer hazır!')

In [ ]:
import shutil

shutil.copy('/content/drive/MyDrive/bert_fake_review_model.pt', './bert_fake_review_model.pt')
shutil.copy('/content/drive/MyDrive/metrikler.pkl', './metrikler.pkl')

print('✅ Model ve metrikler kopyalandı!')

In [ ]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
tokenizer.save_pretrained('./tokenizer')

print('✅ Tokenizer kaydedildi!')

# Kontrol
import os
print(os.listdir('./tokenizer'))

In [ ]:
from google.colab import files
import shutil

shutil.make_archive('tokenizer', 'zip', '.', 'tokenizer')

files.download('bert_fake_review_model.pt')
files.download('metrikler.pkl')
files.download('tokenizer.zip')

In [ ]:
import pandas as pd

df = pd.read_csv('temiz_veri.csv')

label_col = 'label' if 'label' in df.columns else df.columns[-1]
text_col  = 'text_' if 'text_' in df.columns else 'text'

sahte  = df[df[label_col]=='CG'].head(10)
gercek = df[df[label_col]=='OR'].head(10)

print("=== SAHTE YORUMLAR (CG) ===")
for i, row in sahte.iterrows():
    print(f"\n[{i}] {row[text_col][:150]}")

print("\n\n=== GERÇEK YORUMLAR (OR) ===")
for i, row in gercek.iterrows():
    print(f"\n[{i}] {row[text_col][:150]}")